# Classification Ages: Young vs Old Battery

> **Authors:** Davide Corso, Marco Soldani  
> **Context:** Analysis of Galvanostatic Electrochemical Impedance Spectroscopy (GEIS) data on Lithium-Ion batteries (LiCoO₂).

>
> **Expected result:** Given a Nyquist diagram at a certain SOC (chosen by the user or randomly), classify whether the battery is new (Epoch 0–2) or old (Epoch 3–4).

---

### Difference with the previous notebook

| | NB1: Leave-One-Out | **NB2: Classification Ages** |
|---|---|---|
| **Type** | Regression | **Classification** |
| **Target** | Z_real, Z_imag | **Young / Old** |
| **Input** | Epoch, Temp, SOC, Freq | **Temp, Freq, Z_real, Z_imag** |
| **Challenge** | Predict one graph | **Distinguish aging from the shape of the Nyquist plot** |


> In this task, impedance is the **input** (not the target): the model must infer from the appearance of the Nyquist plot whether the battery is young or old.

# 1. Setup

In [3]:
import sys
assert sys.version_info >= (3, 5), "Python 3.5+ is required"

# --- STANDARD LIBRARIES ---
import warnings
import time

# --- SCIENTIFIC LIBRARIES ---
import numpy as np
import pandas as pd
import scipy.io

# --- VISUALIZATION ---
import matplotlib.pyplot as plt
import matplotlib as mpl
import seaborn as sns

# --- MACHINE LEARNING ---
from sklearn.model_selection import (
    train_test_split,
    cross_val_score,
    GridSearchCV,
    StratifiedKFold
)
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import (
    RandomForestClassifier,
    GradientBoostingClassifier,
    BaggingClassifier,
    ExtraTreesClassifier
)
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
    roc_curve,
    roc_auc_score
)

# --- CONFIGURATION ---
warnings.filterwarnings("ignore")

plt.style.use("seaborn-v0_8-whitegrid")
mpl.rc("axes", labelsize=13)
mpl.rc("xtick", labelsize=11)
mpl.rc("ytick", labelsize=11)
mpl.rc("legend", fontsize=10)
mpl.rc("figure", dpi=120)

# --- CONSTANTS ---
TEMP_COLORS = {
    20.0: "#1b9e77",
    22.5: "#d95f02",
    25.0: "#7570b3",
    27.5: "#e7298a",
    30.0: "#66a61e",
    35.0: "#e6ab02",
    40.0: "#a6761d",
    47.5: "#666666",
}

EPOCH_LABELS = {
    0: "Epoch 0 (Fresh)",
    1: "Epoch 1",
    2: "Epoch 2",
    3: "Epoch 3",
    4: "Epoch 4 (Aged)",
}

SOC_MARKERS = {
    0: "o",
    1: "s",
    2: "^",
    3: "D",
    4: "v",
}

TEMPS_SORTED = sorted(TEMP_COLORS.keys())
EPOCHS = list(EPOCH_LABELS.keys())

CLASS_NAMES = ["Young (E0–E2)", "Old (E3–E4)"]
CLASS_COLORS = ["#2ecc71", "#e74c3c"]

# --- ENVIRONMENT INFO ---
def print_environment_info():
    import sklearn

    print("Setup completed")
    print(
        f"Python {sys.version_info.major}.{sys.version_info.minor} | "
        f"NumPy {np.__version__} | "
        f"Pandas {pd.__version__} | "
        f"Scikit-learn {sklearn.__version__}"
    )

print_environment_info()

Setup completed
Python 3.12 | NumPy 2.0.1 | Pandas 2.3.3 | Scikit-learn 1.7.1


# 2. Data Loading

In [6]:
# --- LOAD DATASET ---
from pathlib import Path

file_path = Path("..") / "data" / "raw" / "GEIS.mat"
mat_data = scipy.io.loadmat(file_path, squeeze_me=True, struct_as_record=False)

# --- TEMPERATURE MAPPING ---
TEMP_KEYS = {
    "GEIS_20": 20.0,
    "GEIS_22_5": 22.5,
    "GEIS_25": 25.0,
    "GEIS_27_5": 27.5,
    "GEIS_30": 30.0,
    "GEIS_35": 35.0,
    "GEIS_40": 40.0,
    "GEIS_47_5": 47.5,
}

# --- DATA EXTRACTION ---
records = []

for epoch in range(5):
    aging_struct = getattr(mat_data, f"Aging{epoch}", mat_data.get(f"Aging{epoch}"))
    
    for temp_key, temp_val in TEMP_KEYS.items():
        data = getattr(aging_struct, temp_key)
        
        for soc_idx, soc_data in enumerate(data):
            for row in soc_data:
                records.append({
                    "Epoch": epoch,
                    "Temperature": temp_val,
                    "SOC": soc_idx,
                    "Frequency": row[0],
                    "Z_real": row[1] * 1000,
                    "Z_imag": row[2] * 1000,
                })

df = pd.DataFrame(records)

# --- CLASSIFICATION LABELS ---
df["Age_class"] = (df["Epoch"] >= 3).astype(int)  # 0 = Young, 1 = Old
df["Age_label"] = df["Age_class"].map({0: "Young", 1: "Old"})

# --- DATASET SUMMARY ---
print(f"Dataset: {df.shape[0]:,} rows")

print("\n=== CLASS DISTRIBUTION ===")
young_count = (df["Age_class"] == 0).sum()
old_count = (df["Age_class"] == 1).sum()

print(f"Young (Epoch 0–2): {young_count:,} samples ({young_count / len(df) * 100:.1f}%)")
print(f"Old   (Epoch 3–4): {old_count:,} samples ({old_count / len(df) * 100:.1f}%)")

Dataset: 9,805 rows

=== CLASS DISTRIBUTION ===
Young (Epoch 0–2): 5,880 samples (60.0%)
Old   (Epoch 3–4): 3,925 samples (40.0%)
